## Load and Inspect

In [1]:
# RIDE SUMMARY CLEANING
#
# Input:  ride_stats.csv (long format: two rows per ride-year, one for average
#         wait and one for maximum wait, plus "all_time" summary rows)
# Output: rides_clean_all_years.csv and rides_clean_2025.csv
#         Columns: ride_id, ride_name, park_name, year, avg_wait, max_wait,
#         category, wait_tier (one row per ride-year)
#
# What this notebook does:
# 1. Drops the all_time rows and archived (retired) attractions. 14 archived
#    attractions had 2025 data (e.g. Rock 'n' Roller Coaster) and were excluded
#    on purpose, since visitors can no longer ride them.
# 2. Reshapes from long to wide so avg_wait and max_wait share a row.
# 3. Validates the reshape (no missing values, max >= avg, no duplicates).
# 4. Adds category and wait_tier.
#
# Notes for later steps:
# - Use rides_clean_2025.csv for Viz 4 and 5. The number of tracked rides grows
#   every year (45 in 2015 to 78 in 2025), so ride comparisons across years
#   would mix different sets of attractions. 2026 is a partial year (through
#   September), so it is not used.
# - category was assigned from keywords in the ride name, with unmatched
#   attractions defaulting to family. It is a team-assigned label, not part of
#   the original data, and a few attractions may still be misfiled. If a ride
#   looks wrong in Viz 4, add a keyword for it in guess_category and rerun.
# - wait_tier cutoffs are short (<20 min), medium (20 to 45), and long (45+),
#   based on avg_wait. Use the same cutoffs when building the Tableau viz
#   extension in Viz 5. A ride at exactly 45 minutes counts as long.
# - Website / design document: mention that archived attractions were excluded,
#   that categories were assigned by the team from ride names, and that
#   average and maximum wait are yearly summaries rather than daily data.

In [2]:
import pandas as pd

stats = pd.read_csv("ride_stats.csv")

stats.head()

,park_id,park_name,year,ride_id,ride_name,is_archived,metric,minutes
0,5,Epcot,all_time,10916,Guardians of the Galaxy: Cosmic Rewind,0,ride_avg_wait,63
1,5,Epcot,all_time,160,Test Track,0,ride_avg_wait,57
2,5,Epcot,all_time,10914,Remy's Ratatouille Adventure,0,ride_avg_wait,56
3,5,Epcot,all_time,2679,Frozen Ever After,0,ride_avg_wait,52
4,5,Epcot,all_time,6701,Meet Anna and Elsa at Royal Sommerhus,0,ride_avg_wait,34


In [3]:
print(stats.shape)
print(stats.isna().sum())
print(stats["metric"].unique())
print(stats["year"].unique())
print(stats["is_archived"].unique())

(2022, 8)
park_id        0
park_name      0
year           0
ride_id        0
ride_name      0
is_archived    0
metric         0
minutes        0
dtype: int64
['ride_avg_wait' 'ride_max_wait']
['all_time' '2015' '2016' '2017' '2018' '2019' '2020' '2021' '2022' '2023'
 '2024' '2025' '2026']
[0 1]


## Drop "all-time" rows and archived rides

In [4]:
stats = stats[stats["year"] != "all_time"].copy()
stats["year"] = stats["year"].astype(int)

stats = stats[stats["is_archived"] == 0]
stats = stats.drop(columns=["park_id", "is_archived"])
stats.shape

(1434, 6)

## Reshape: one row per ride-year

In [5]:
# 3. Reshape from long to wide: the raw file has two rows per ride-year (one
# per metric), but our plots need avg_wait and max_wait in the same row.
# pivot_table makes each metric its own column, giving one row per ride-year.

rides = (
    stats.pivot_table(index=["ride_id", "ride_name", "park_name", "year"],
                      columns="metric", values="minutes", aggfunc="first")
    .reset_index()
)
rides.columns.name = None
rides = rides.rename(columns={"ride_avg_wait": "avg_wait", "ride_max_wait": "max_wait"})
rides.head()

,ride_id,ride_name,park_name,year,avg_wait,max_wait
0,110,Expedition Everest - Legend of the Forbidden M...,Animal Kingdom,2015,29,55
1,110,Expedition Everest - Legend of the Forbidden M...,Animal Kingdom,2016,30,57
2,110,Expedition Everest - Legend of the Forbidden M...,Animal Kingdom,2017,36,67
3,110,Expedition Everest - Legend of the Forbidden M...,Animal Kingdom,2018,39,72
4,110,Expedition Everest - Legend of the Forbidden M...,Animal Kingdom,2019,41,75


## Validation

In [6]:
# 4. Validate the reshape: every ride-year should have both an avg_wait and a
# max_wait (no missing values), max should never be below avg, and no ride-year
# should appear twice. The last line shows how many rides are tracked per year,
# which supports using 2025 for the ride comparisons.

print(rides.isna().sum())
print("max < avg:", (rides["max_wait"] < rides["avg_wait"]).sum())
print("duplicates:", rides.duplicated(["ride_id", "year"]).sum())
print(rides.groupby("year")["ride_id"].nunique())

ride_id      0
ride_name    0
park_name    0
year         0
avg_wait     0
max_wait     0
dtype: int64
max < avg: 0
duplicates: 0
year
2015    45
2016    51
2017    45
2018    46
2019    59
2020    60
2021    50
2022    61
2023    70
2024    73
2025    78
2026    79
Name: ride_id, dtype: int64


In [7]:
raw = pd.read_csv("ride_stats.csv")
archived = raw[(raw["is_archived"] == 1) & (raw["year"] == "2025")]
print(archived["ride_id"].nunique())
print(archived[["ride_name", "park_name"]].drop_duplicates())

14
                                              ride_name          park_name
332                              Soarin' Across America              Epcot
1181  Meet Santa Jack and Sally at Town Square Theat...      Magic Kingdom
1193                           Liberty Square Riverboat      Magic Kingdom
1196          Walt Disney World Railroad - Frontierland      Magic Kingdom
1210                                  Tom Sawyer Island      Magic Kingdom
1677  Meet Stitch at Celebrity Spotlight at Disney J...  Hollywood Studios
1678         Rock 'n' Roller Coaster Starring Aerosmith  Hollywood Studios
1682        Star Wars Launch Bay: Encounter Darth Vader  Hollywood Studios
1685               Star Wars Launch Bay: Meet Chewbacca  Hollywood Studios
1689       Star Wars Launch Bay: BB-8 Astromech on Duty  Hollywood Studios
1691                                   Muppet*Vision 3D  Hollywood Studios
1983                                           DINOSAUR     Animal Kingdom
1984                  

## Add category and wait tier

In [8]:
# 5. Add category and wait tier
# category: label from keywords in the ride name; anything unmatched defaults
# to family.
# wait_tier: bins avg_wait into short (<20 min), medium (20 to 45), and long
# (45+) for the Viz 5 park comparison. Match these cutoffs in Tableau.

# AI usage note: I used Claude (Anthropic) to help come up with the keyword
# lists below for assigning attraction categories. I reviewed the resulting
# labels against the 2025 ride list and adjusted the keywords where they
# misfiled rides.

def guess_category(name):
    n = name.lower()
    if any(w in n for w in ["meet", "greet", "character", "encounter", "astromech"]):
        return "meet-and-greet"
    if any(w in n for w in ["show", "theater", "theatre", "stage", "parade", "fireworks",
                            "laugh floor", "turtle talk", "musical", "zootopia",
                            "enchanted tales", "muppet"]):
        return "show"
    if any(w in n for w in ["walk", "trail", "exhibit", "experience",
                            "journey of water"]):
        return "walk-through"
    if any(w in n for w in ["mountain", "coaster", "terror", "rock 'n'", "test track",
                            "everest", "flight of passage", "guardians", "resistance",
                            "mine train", "barnstormer", "mission: space", "tiana",
                            "kali river"]):
        return "thrill"
    return "family"

rides["category"] = rides["ride_name"].apply(guess_category)
rides["wait_tier"] = pd.cut(rides["avg_wait"], bins=[-float("inf"), 20, 45, float("inf")],
                            labels=["short", "medium", "long"], right=False)
print(rides.drop_duplicates("ride_id")["category"].value_counts())

# 6. Save the cleaned files
rides.to_csv("rides_clean_all_years.csv", index=False)
rides_2025 = rides[rides["year"] == 2025]
rides_2025.to_csv("rides_clean_2025.csv", index=False)
print(len(rides_2025))

category
family            47
meet-and-greet    15
thrill            13
show               7
walk-through       2
Name: count, dtype: int64
78
